# A-Network — 物理神经网络训练

基于 PyTorch 的 3D 张量场（80×80×80）神经网络。
Token 注入场中，经 20 步 26-邻域扩散后读出为 logits。

**仓库**: https://github.com/Mn3TR/a-network

权重和日志自动保存到 Google Drive（`a-network/` 目录）。

## 1. 环境

In [ ]:
# 回到 /content 再删（避免删掉当前目录导致 shell 挂掉）
%cd /content
!rm -rf /content/a-network
!git clone https://github.com/Mn3TR/a-network.git
%cd a-network
!pip install -q torch tokenizers numpy matplotlib datasets
import torch
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/a-network'
!mkdir -p {OUTPUT_DIR}/output {OUTPUT_DIR}/log
print(f'输出目录: {OUTPUT_DIR}')

## 2. 配置

In [ ]:
import sys, importlib

# 清除旧缓存，确保用最新代码
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]
!rm -rf src/anetwork/__pycache__
sys.path.insert(0, '.')

from src.anetwork.config import ANetworkConfig
from src.anetwork.model import ANetwork
from src.anetwork.tokenizer import TokenizerWrapper

TRAIN_EPOCHS = 5
LEARNING_RATE = 1e-4
GRAD_ACCUM = 4
BATCH_SIZE = 8
DATASET = 'roneneldan/TinyStories'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = TokenizerWrapper('tokenizer/tokenizer.json')
print(f'Vocab: {tokenizer.vocab_size}')

a_cfg = ANetworkConfig(vocab_size=tokenizer.vocab_size)
net = ANetwork(a_cfg, device=DEVICE).to(DEVICE)
print(f'Params: {sum(p.numel() for p in net.parameters()):,}')

## 3. 加载数据

In [ ]:
from src.anetwork.data import load_data, pack_batch, count_batch_steps
tokens = load_data(DATASET, tokenizer)
total_steps = count_batch_steps(tokens, BATCH_SIZE)
print(f'Tokens: {len(tokens):,}  Steps/epoch: {total_steps:,}  B={BATCH_SIZE}')

## 4. 训练

In [ ]:
import math, time
B, N = BATCH_SIZE, a_cfg.N
optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
for epoch in range(TRAIN_EPOCHS):
    if TRAIN_EPOCHS > 1:
        p = epoch / (TRAIN_EPOCHS - 1)
        lr = float(1e-6 + (LEARNING_RATE - 1e-6) * (1 + math.cos(math.pi * p)) * 0.5)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
    net.train(); epoch_loss = 0.0; epoch_start = time.time(); optimizer.zero_grad()
    fields = torch.zeros(B, N, device=DEVICE)
    incomings = torch.zeros(B, N, device=DEVICE)
    for step, (inp, tgt) in enumerate(pack_batch(tokens, B)):
        loss, fields, incomings = net.train_step_batch(
            fields, incomings,
            torch.tensor(inp, device=DEVICE),
            torch.tensor(tgt, device=DEVICE))
        loss.backward(); epoch_loss += loss.item()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
            optimizer.step(); optimizer.zero_grad()
        if (step + 1) % max(1, total_steps // 10) == 0:
            print(f'  e{epoch} [{step+1}/{total_steps}] loss={loss.item():.4f}')
    if total_steps % GRAD_ACCUM:
        optimizer.step(); optimizer.zero_grad()
    avg_loss = epoch_loss / total_steps
    print(f'Epoch {epoch}: loss={avg_loss:.4f}  time={time.time()-epoch_start:.0f}s  lr={lr:.8f}')
    net.save(f'{OUTPUT_DIR}/output/weights_e{epoch}.pt')
print('Done!')

## 5. 生成

In [ ]:
net.eval()
seed = tokenizer.encode('Time')[:3]
gen = net.generate(seed, max_tokens=50)
print(tokenizer.decode(gen))

## 6. 场可视化

In [ ]:
import numpy as np, matplotlib.pyplot as plt
f = net.field.detach().cpu().numpy().reshape(80, 80, 80)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
for i, z in enumerate([20, 40, 60]):
    im = ax[i].imshow(f[:, :, z], cmap='viridis')
    ax[i].set_title(f'z={z}'); ax[i].axis('off')
fig.colorbar(im, ax=ax, shrink=0.6); plt.show()